##### Imports

In [1]:
import numpy as np
import pandas as pd
import joblib
from pathlib import Path
from econml.dml import CausalForestDML

from gg570_d200.auxiliary_functions.forest_riesz_funcs import call_forestriesz, call_forestriesz_cross
from gg570_d200.auxiliary_functions.ate_estimation_funcs import forest_riesz_gate, causal_dml_gate

IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html


In [2]:
root = Path.cwd().parent
data_path = root / "data"

In [3]:
np.random.seed(21)

In [4]:
df_scaled = pd.read_csv(data_path / "df_scaled.csv")

In [5]:
saved_joblib = joblib.load(data_path / "covariate_scaler.joblib")
scaler = saved_joblib['scaler']

In [6]:
df = pd.read_csv(data_path / "df.csv")

In [7]:
treatment_col = 'Training (last 3 months)'
outcome_col = 'Underemployment hours'
covariate_cols = [col for col in df_scaled.columns if col not in treatment_col and col not in outcome_col and col not in ['prop_scores']]

---

##### Key specification

In [9]:
cross_scale, est_cs = call_forestriesz_cross(df_scaled, covariate_cols, treatment_col, outcome_col, methods=['dr', 'plugin'], return_est=True)
cross_scale

{'dr': {'est': -0.16881398495582658,
  'low': -1.3646790706040013,
  'high': 1.027051100692348,
  'p_val': 0.7820275135261565},
 'plugin': {'est': -0.03872457663475174,
  'low': -0.4595074374118259,
  'high': 0.3820582841423225,
  'p_val': 0.8568580545187574}}

In [19]:
for est_cs_i in est_cs:
    heterogeneity_importance = est_cs_i.feature_importances_
    top_3_ids = np.argsort(heterogeneity_importance)[-3:][::-1]
    for id in top_3_ids:
        print(f"{covariate_cols[id]}: {heterogeneity_importance[id]}")
    print("")

LKWFWM_bin_(12.2, 15.0]: 0.17954471488135304
LKWFWM_bin_(6.6, 9.4]: 0.11011751478054921
FTPTWK_bin_(0.999, 1.5]: 0.10184970032114685

LKWFWM_bin_(12.2, 15.0]: 0.35017268036330124
FTPTWK_bin_(0.999, 1.5]: 0.10694748384173042
AGE: 0.06844316466172358

LKWFWM_bin_(12.2, 15.0]: 0.38029891505287994
LKWFWM_bin_(6.6, 9.4]: 0.14812841856000827
FTPTWK_bin_(0.999, 1.5]: 0.045850310178583514



In [22]:
mask = df_scaled['FTPTWK_bin_(0.999, 1.5]'] > 0.0
for est_cs_i in est_cs:
    print(forest_riesz_gate(df_scaled, covariate_cols, treatment_col, outcome_col, est_cs_i, mask))
    print(forest_riesz_gate(df_scaled, covariate_cols, treatment_col, outcome_col, est_cs_i, ~mask))
    print()

[0.4697139158660449, -0.48805698516437607, 1.4274848168964658, 0.3364448098463888]
[-0.9502976580102409, -1.7967123969045637, -0.10388291911591796, 0.02777030529054003]

[0.46165400499161724, -0.5641600490812247, 1.4874680590644593, 0.3777466396688465]
[-0.8926602526956635, -1.709317730679989, -0.07600277471133798, 0.03216377271648896]

[0.6157854837261798, -0.3860716146057068, 1.6176425820580664, 0.22832679770819464]
[-0.9466252651622133, -1.744675893875832, -0.1485746364485947, 0.020079752955481478]



---

##### Alternative specifications

###### ForestRiesz alternatives

In [ ]:
no_cross_scale, est_ncs = call_forestriesz(df_scaled, covariate_cols, treatment_col, outcome_col, methods=['dr', 'plugin'], return_est=True)
no_cross_scale

{'dr': {'est': -0.25472215412176613,
  'low': -0.862982411199671,
  'high': 0.35353810295613874,
  'p_val': 0.41177309656265804},
 'plugin': {'est': -0.0367478621002449,
  'low': -0.296532478464972,
  'high': 0.22303675426448222,
  'p_val': 0.7815905097974638}}

In [ ]:
heterogeneity_importance = est_ncs.feature_importances_

In [65]:
top_3_ids = np.argsort(heterogeneity_importance)[-3:][::-1]
for id in top_3_ids:
    print(f"{covariate_cols[id]}: {heterogeneity_importance[id]}")

LKWFWM_bin_(12.2, 15.0]: 0.4264092363788278
FTPTWK_bin_(1.5, 2.0]: 0.0867071098779133
FTPTWK_bin_(0.999, 1.5]: 0.07434881656949945


In [ ]:
mask = df_scaled[f'LKWFWM_bin_(12.2, 15.0]'] > 0.0
print(forest_riesz_gate(df_scaled, covariate_cols, treatment_col, outcome_col, est_ncs, mask))
print(forest_riesz_gate(df_scaled, covariate_cols, treatment_col, outcome_col, est_ncs, ~mask))

[-0.06610712231675207, -0.7141819729454321, 0.5819677283119279, 0.8415377724871507]
[-0.9999364464494216, -2.589237235223907, 0.5893643423250634, 0.21752141262800118]


In [ ]:
mask = df_scaled['FTPTWK_bin_(1.5, 2.0]'] > 0.0
print(forest_riesz_gate(df_scaled, covariate_cols, treatment_col, outcome_col, est_ncs, mask))
print(forest_riesz_gate(df_scaled, covariate_cols, treatment_col, outcome_col, est_ncs, ~mask))

[-0.9734022195622221, -1.7571872671471378, -0.18961717197730632, 0.014927706648095063]
[0.5403972926272679, -0.40170298143228267, 1.4824975666868183, 0.26090563380196885]


---

In [ ]:
no_cross_no_scale, est_ncns = call_forestriesz(df, covariate_cols, treatment_col, outcome_col, methods=['dr', 'plugin'], return_est=True)
no_cross_no_scale

{'dr': {'est': -0.25635841308440976,
  'low': -0.864829870931797,
  'high': 0.35211304476297745,
  'p_val': 0.40893850907654183},
 'plugin': {'est': -0.037142018151062196,
  'low': -0.2968813883836602,
  'high': 0.22259735208153578,
  'p_val': 0.7792707652518849}}

In [ ]:
heterogeneity_importance = est_ncns.feature_importances_

In [57]:
top_3_ids = np.argsort(heterogeneity_importance)[-3:][::-1]
for id in top_3_ids:
    print(f"{covariate_cols[id]}: {heterogeneity_importance[id]}")

LKWFWM_bin_(12.2, 15.0]: 0.4263337474119389
FTPTWK_bin_(1.5, 2.0]: 0.08675611041819323
FTPTWK_bin_(0.999, 1.5]: 0.07430127987237045


In [ ]:
mask = df[f'LKWFWM_bin_(12.2, 15.0]'] > 0.0
print(forest_riesz_gate(df, covariate_cols, treatment_col, outcome_col, est_ncns, mask))
print(forest_riesz_gate(df, covariate_cols, treatment_col, outcome_col, est_ncns, ~mask))

[-0.0681012581421217, -0.7163113054986712, 0.5801087892144277, 0.8368574994000189]
[-1.0001587409446262, -2.5905959470258813, 0.5902784651366286, 0.21774798881575674]


In [ ]:
mask = df[f'FTPTWK_bin_(1.5, 2.0]'] > 0.0
print(forest_riesz_gate(df, covariate_cols, treatment_col, outcome_col, est_ncns, mask))
print(forest_riesz_gate(df, covariate_cols, treatment_col, outcome_col, est_ncns, ~mask))

[-0.9747438326397199, -1.758704852837987, -0.1907828124414528, 0.014812302091380847]
[0.5384350490096292, -0.4040941898801421, 1.4809642878994005, 0.2628582169547924]


---

In [44]:
cross_no_scale = call_forestriesz_cross(df, covariate_cols, treatment_col, outcome_col, methods=['dr', 'plugin'])
cross_no_scale

{'dr': {'est': -0.16996809045336814,
  'low': -1.366777810083686,
  'high': 1.0268416291769498,
  'p_val': 0.7807441141944769},
 'plugin': {'est': -0.03936654703911537,
  'low': -0.460119042457307,
  'high': 0.3813859483790763,
  'p_val': 0.8545009456647903}}

---

###### CausalForestDML

In [45]:
causal_dml = CausalForestDML(
    discrete_treatment=True,
    criterion='mse',
    n_estimators=100,
    min_samples_leaf=2,
    min_var_fraction_leaf=0.001,
    min_var_leaf_on_val=True,
    min_impurity_decrease=0.01,
    #max_samples=.8,
    max_depth=None,
    inference=True, #inference=False
    subforest_size=2, #subforest_size=1
    honest=True,
    verbose=0,
    n_jobs=-2,
    random_state=21,
    cv=3
)

In [46]:
causal_dml.fit(df_scaled[outcome_col], df_scaled[treatment_col], X=df_scaled[covariate_cols])
causal_dml.ate_, causal_dml.ate_stderr_

(array([-0.03379913]), array([0.42109106]))

In [47]:
heterogeneity_importance = causal_dml.feature_importances_

In [48]:
top_3_ids = np.argsort(heterogeneity_importance)[-3:][::-1]
for id in top_3_ids:
    print(f"{covariate_cols[id]}: {heterogeneity_importance[id]}")

AGE: 0.12016064204929525
REFWKD: 0.06559192860626194
HIQUAL22: 0.05090186012900608


In [49]:
aage_bins = ["(1.99, 4.0]", "(10.0, 12.0]", "(4.0, 6.0]", "(6.0, 8.0]", "(8.0, 10.0]"]

In [ ]:
for bin in aage_bins:
    mask = df_scaled[f'AAGE_bin_{bin}'] > 0.0
    print(bin)
    print(causal_dml_gate(df_scaled, covariate_cols, causal_dml, mask))
    print(causal_dml_gate(df_scaled, covariate_cols, causal_dml, ~mask),"\n")

[0.4439220481437849, -4.238005028477581, 5.12584912476515, 0.8525732674546236]
[0.43759330543594094, -5.0355959290684105, 5.910782539940292, 0.8754786424206622] 

[0.897982680518652, -7.889935927161423, 9.685901288198725, 0.8412643287892894]
[0.35747163564126666, -4.230781133585723, 4.9457244048682565, 0.8786339440360031] 

[0.2679351397235288, -4.185432863851554, 4.721303143298612, 0.9061306779659224]
[0.48565221593514063, -5.180829157094368, 6.152133588964648, 0.8665982758016404] 

[0.3244058040060629, -4.18400889770629, 4.8328205057184155, 0.8878458979772033]
[0.48432692206265154, -5.271650660785073, 6.240304504910374, 0.8690086594137885] 

[0.44230010773247985, -4.308591924174136, 5.193192139639096, 0.8552143890995381]
[0.4363769277624678, -5.223824119385391, 6.096577974910326, 0.8798931137315333] 



In [ ]:
df_scaled['REFWKD_bins'] = pd.qcut(df_scaled['REFWKD'], q=5, duplicates='drop')
refwkd_bins = df_scaled['REFWKD_bins'].cat.categories

for bin_label in refwkd_bins:
    mask = df_scaled['REFWKD_bins'] == bin_label
    print(bin_label)
    print(causal_dml_gate(df_scaled, covariate_cols, causal_dml, mask))
    print(causal_dml_gate(df_scaled, covariate_cols, causal_dml, ~mask),"\n")

(-1.64, -1.082]
[0.42587022359288473, -4.710686186352758, 5.562426633538526, 0.8709121461022447]
[0.44125835210343967, -5.057361553209365, 5.939878257416241, 0.8750202347656462] 

(-1.082, -0.303]
[0.4113043899379888, -4.9971892751234686, 5.819798054999445, 0.8815133316054495]
[0.44549428906898647, -4.983193147606602, 5.874181725744573, 0.8722190318657961] 

(-0.303, 0.365]
[0.4176196440038939, -5.343680253801295, 6.178919541809082, 0.8870231260110737]
[0.4422282745986287, -4.909741136422178, 5.794197685619434, 0.8713452215571271] 

(0.365, 1.033]
[0.4691867034825747, -5.104514015884092, 6.042887422849241, 0.8689540900194856]
[0.4296571287610749, -4.953860123733794, 5.813174381255943, 0.8756986457487133] 

(1.033, 1.701]
[0.46630647411051473, -4.80099047008072, 5.733603418301748, 0.8622482998211343]
[0.4314785952562868, -5.028410295968789, 5.891367486481362, 0.8769079916568943] 



In [60]:
df_scaled['HIQUAL22_bins'] = pd.qcut(df_scaled['HIQUAL22'], q=5, duplicates='drop')
hiqual22_bins = df_scaled['HIQUAL22_bins'].cat.categories

for bin_label in hiqual22_bins:
    mask = df_scaled['HIQUAL22_bins'] == bin_label
    print(bin_label)
    print(causal_dml_gate(df_scaled, covariate_cols, causal_dml, mask))
    print(causal_dml_gate(df_scaled, covariate_cols, causal_dml, ~mask),"\n")

(-1.1769999999999998, -0.848]
[-0.31858824545105846, -4.996984750210298, 4.3598082593081795, 0.8938223806135186]
[0.06500036056709617, -4.598281828355337, 4.7282825494895295, 0.9782049333511509] 

(-0.848, -0.68]
[-0.2605538303047884, -4.589410382080641, 4.068302721471063, 0.9060912607272438]
[-0.0814713305308577, -4.7547250831602215, 4.591782422098506, 0.9727423069762695] 

(-0.68, 0.04]
[-0.02207988373056389, -4.623927323047323, 4.579767555586194, 0.9924968035616082]
[-0.10000586058149799, -4.786816302107778, 4.586804580944781, 0.9666412604693542] 

(0.04, 1.256]
[0.038510615253818, -4.324986024408915, 4.40200725491655, 0.9861989443838624]
[-0.13971363960494965, -4.942496706803025, 4.663069427593125, 0.9545327445472578] 

(1.256, 2.191]
[0.46626520721410236, -5.446103011674126, 6.37863342610233, 0.8771618880505727]
[-0.12907502254965303, -4.680474973293997, 4.42232492819469, 0.9556736694271242] 



In [70]:
mask = df_scaled[f'FTPTWK_bin_(1.5, 2.0]'] > 0.0
print(causal_dml_gate(df_scaled, covariate_cols, causal_dml, mask))
print(causal_dml_gate(df_scaled, covariate_cols, causal_dml, ~mask))

[-0.1578066923088366, -4.897030370299807, 4.581416985682132, 0.9479646888394797]
[-0.0016853489332691828, -4.592055189909651, 4.588684492043112, 0.9994258435236247]


In [ ]:
target = "FTPTWK_bin_(1.5, 2.0]"

high_corr = (
    df_scaled.corr(numeric_only=True)[target]
    .drop(target)
    .sort_values(key=np.abs, ascending=False)
)
high_corr[high_corr.abs() >= 0.5]

---

In [17]:
max_col_mean = scaler.mean_[max_id]
max_col_std = scaler.scale_[max_id]

df_scaled[f"{max_col}_original"] = df_scaled[max_col] * max_col_std + max_col_mean

DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`


In [19]:
df_scaled

,"AAGE_bin_(1.99, 4.0]","AAGE_bin_(10.0, 12.0]","AAGE_bin_(4.0, 6.0]","AAGE_bin_(6.0, 8.0]","AAGE_bin_(8.0, 10.0]",AGE,"ANXIOUS_bin_(-0.01, 2.0]","ANXIOUS_bin_(2.0, 4.0]","ANXIOUS_bin_(4.0, 6.0]","ANXIOUS_bin_(6.0, 8.0]",...,"edageband_bin_(10.916, 27.8]","edageband_bin_(27.8, 44.6]","edageband_bin_(44.6, 61.4]","edageband_bin_(61.4, 78.2]","edageband_bin_(78.2, 95.0]",Training (last 3 months),Underemployment hours,prop_scores,AGE_original,AGE_original_bin
0,-0.266552,-0.418452,-0.529246,1.566203,-0.617794,0.197454,-0.884804,-0.487591,2.034686,-0.399483,...,0.165732,-0.104933,-0.022255,0.0,-0.124843,1,10.0,0.305241,44.0,"(42.0, 45.0]"
1,-0.266552,-0.418452,-0.529246,1.566203,-0.617794,-0.247202,1.130194,-0.487591,-0.491476,-0.399483,...,0.165732,-0.104933,-0.022255,0.0,-0.124843,0,5.0,0.550512,39.0,"(38.0, 42.0]"
2,-0.266552,-0.418452,-0.529246,-0.638487,1.618664,0.997835,1.130194,-0.487591,-0.491476,-0.399483,...,0.165732,-0.104933,-0.022255,0.0,-0.124843,0,3.0,0.211601,53.0,"(49.0, 53.0]"
3,-0.266552,-0.418452,1.889480,-0.638487,-0.617794,-1.136514,-0.884804,-0.487591,2.034686,-0.399483,...,0.165732,-0.104933,-0.022255,0.0,-0.124843,0,5.0,0.268558,29.0,"(27.0, 31.0]"
4,-0.266552,-0.418452,1.889480,-0.638487,-0.617794,-1.225446,1.130194,-0.487591,-0.491476,-0.399483,...,0.165732,-0.104933,-0.022255,0.0,-0.124843,0,8.0,0.179623,28.0,"(27.0, 31.0]"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2015,-0.266552,-0.418452,-0.529246,-0.638487,1.618664,0.908903,-0.884804,-0.487591,-0.491476,2.503235,...,0.165732,-0.104933,-0.022255,0.0,-0.124843,1,7.0,0.455386,52.0,"(49.0, 53.0]"
2016,-0.266552,-0.418452,1.889480,-0.638487,-0.617794,-1.403308,-0.884804,-0.487591,2.034686,-0.399483,...,0.165732,-0.104933,-0.022255,0.0,-0.124843,0,2.0,0.112617,26.0,"(16.999, 27.0]"
2017,-0.266552,2.389762,-0.529246,-0.638487,-0.617794,1.442491,1.130194,-0.487591,-0.491476,-0.399483,...,0.165732,-0.104933,-0.022255,0.0,-0.124843,0,8.0,0.036216,58.0,"(57.0, 64.0]"
2018,-0.266552,-0.418452,-0.529246,-0.638487,1.618664,0.553179,1.130194,-0.487591,-0.491476,-0.399483,...,0.165732,-0.104933,-0.022255,0.0,-0.124843,1,20.0,0.425192,48.0,"(45.0, 49.0]"


In [21]:
ate = causal_dml.ate(X=X_group)
lb, ub = causal_dml.ate_interval(X=X_group, alpha=0.05)

ate, (lb, ub)